# 01 Cargar 65 originales en HDFS (3.2)

Comprueba SHA-256 antes/después; no sobrescribe existentes y no toca proyecto_afp antiguo.

In [ ]:
from pathlib import Path
import sys
BASE = Path.home()/'BigData_UPAO'/'proyecto_afp_v2'
sys.path.insert(0,str(BASE/'src'))
from afp_config import raiz_fuentes, ruta_hdfs, EVIDENCIAS, HDFS
from afp_tools import ejecutar
EVIDENCIAS.mkdir(parents=True, exist_ok=True)
print('Proyecto:',BASE)
print('Raíz Drive:',raiz_fuentes())
print('HDFS V2:',HDFS)

In [ ]:
from datetime import date
from afp_tools import slug, sha256_local, subir_si_no_existe, escribir_jsonl, sha256_hdfs, existe_hdfs
FUENTES=raiz_fuentes()
ARCHIVOS=sorted(p for p in FUENTES.rglob('*') if p.is_file())
print('Archivos:',len(ARCHIVOS))
if len(ARCHIVOS)!=65:
 raise RuntimeError('Esperados: 65 (7 estructurados + 5 JSON + 48 memorias + 5 controles). Revisa los archivos locales.')
for z in ('raw','staging','processed','quality','catalog'):
 ejecutar(['hdfs','dfs','-mkdir','-p',ruta_hdfs(z)])
print('Se crearon las zonas.')

In [ ]:
INGEST_DATE=date.today().isoformat()
def destino_raw(p):
 rel=p.relative_to(FUENTES)
 carpeta=rel.parts[0]
 if carpeta=='Datos Estructurados':
  familia='sbs' if p.suffix.lower() in ('.xls','.xlsx') else 'bcrp/csv'
  conjunto=slug(p.stem)
 elif carpeta=='Datos Semiestructurados':
  familia='bcrp/json'; conjunto=slug(p.stem)
 elif carpeta=='Datos No estructurados':
  familia='afp_memorias/'+slug(rel.parts[1]); conjunto='memorias_anuales'
 else:
  familia='control'; conjunto=slug(rel.parts[1] if len(rel.parts)>1 else 'registro_descargas')
 return ruta_hdfs('raw',familia,conjunto,f'ingest_date={INGEST_DATE}',p.name)

manifiesto = []

for n, p in enumerate(ARCHIVOS, 1):
    destino = destino_raw(p)

    print(f"\nArchivo {n}/{len(ARCHIVOS)}")
    print("ORIGEN:", repr(str(p)))
    print("EXISTE:", p.exists())
    print("DESTINO:", destino)

    if not p.is_file():
        raise FileNotFoundError(f"No existe: {p}")

    estado = subir_si_no_existe(p, destino)

    manifiesto.append({
        "origen_relativo": str(p.relative_to(FUENTES)).replace("\\", "/"),
        "hdfs_path": destino,
        "bytes": p.stat().st_size,
        "sha256": sha256_local(p),
        "ingest_date": INGEST_DATE,
        "estado": estado
    })

    if n % 10 == 0 or n == len(ARCHIVOS):
        print(f"{n}/{len(ARCHIVOS)} verificados")

local = EVIDENCIAS / f"01_manifiesto_ingesta_{INGEST_DATE}.jsonl"

escribir_jsonl(local, manifiesto)

remoto = ruta_hdfs("raw", "_manifests", local.name)

subir_si_no_existe(local, remoto)

print("\nTOTAL:", len(manifiesto), "archivos")
print("BYTES:", sum(x["bytes"] for x in manifiesto))
print("MANIFIESTO:", remoto)

print(ejecutar([
    "hdfs", "dfs", "-du", "-s", "-h", ruta_hdfs("raw")
]).stdout)

**Guarda captura** del total de 65 archivos, bytes, ruta del manifiesto y de `hdfs dfs -ls /user/<usuario>/datalake_afp_v2/raw`.